**1. Mengumpulkan dan Menganalisis Berbagai Permasalahan (*Problem Discovery*)**

Berdasarkan tinjauan terhadap kondisi di lapangan dan literatur pendukung, terdapat beberapa permasalahan krisis terkait kesehatan mental mahasiswa yang saling berkaitan:

*   **Tingginya Tekanan Akademik dan Gaya Hidup Tidak Sehat:** Mahasiswa sering mengalami stres dan *burnout* akibat tuntutan beban akademik yang tinggi, kompetisi, serta manajemen waktu yang buruk. Hal ini diperparah oleh pola hidup tidak sehat, seperti kurangnya aktivitas fisik, *screen time* yang berlebihan, dan kualitas tidur yang buruk. Studi literatur menunjukkan bahwa kualitas tidur yang buruk dan tingkat aktivitas fisik yang rendah sangat lazim ditemui pada mahasiswa dan memiliki hubungan yang signifikan dengan *burnout* akademik. 
*   **Krisis Kesehatan Mental Skala Nasional:** Krisis kesehatan mental di kalangan generasi muda telah mencapai tingkat yang mengkhawatirkan. Sekitar 15,5 juta remaja Indonesia (usia 15-24 tahun) mengalami gangguan kecemasan dan depresi, namun hanya 9% yang mendapatkan akses ke layanan psikolog profesional. Survei spesifik di lingkup universitas, seperti di UNESA, menemukan bahwa 21,9% mahasiswa berada dalam kelompok risiko tinggi kesehatan mental.
*   **Keterbatasan Akses Layanan Profesional:** Terdapat hambatan besar dalam deteksi dini dan penanganan kesehatan mental. Rasio psikolog di Indonesia baru mencapai 1:200.000 penduduk, sangat jauh dari standar ideal WHO (1:30.000). Selain itu, biaya konseling swasta yang mencapai Rp800.000 per sesi menjadikannya tidak terjangkau bagi mayoritas mahasiswa.
*   **Ketidakmampuan Mengukur Risiko Secara Objektif:** Banyak mahasiswa yang tidak menyadari bahwa kebiasaan sehari-hari mereka berdampak langsung pada kesehatan mental. Berdasarkan hasil analisis data eksploratif (EDA) awal, terlihat jelas bahwa mahasiswa dengan tingkat stres *High* memiliki rata-rata waktu tidur yang lebih rendah (5,88 jam) dan *screen time* yang jauh lebih tinggi (8,65 jam/hari) dibandingkan kelompok stres *Low*. Sayangnya, mahasiswa tidak memiliki alat bantu mandiri yang objektif dan berbasis data untuk memantau indikator gaya hidup ini.

**2. Menentukan Satu Solusi Utama yang Akan Dikembangkan**

Berangkat dari analisis permasalahan di atas, solusi utama yang akan dikembangkan dalam proyek ini adalah **BurnoutGuard: Sistem Deteksi Dini Risiko Kesehatan Mental Mahasiswa Berbasis Artificial Intelligence**. 

Proyek ini diposisikan sebagai lapisan pertahanan pertama (*first layer of defense*) yang proaktif, murah, dan mandiri. Sebagai bagian dari peran *Data Science* dalam pengembangan *BurnoutGuard*, solusi ini dirancang dengan cakupan berikut:

1.  **Pendekatan Berbasis Data Gaya Hidup (*Lifestyle-Driven*):** Membangun solusi yang memprediksi tingkat stres (Rendah, Sedang, Tinggi) menggunakan input data gaya hidup sehari-hari yang mudah diukur oleh mahasiswa itu sendiri, seperti durasi tidur, *screen time*, beban tugas, dan frekuensi ujian. Data ini dipilih karena terbukti secara statistik memiliki korelasi kuat terhadap tingkat stres berdasarkan proses *Exploratory Data Analysis* (EDA).
2.  **Pembuatan *Dashboard* Analitik Interaktif:** Mengembangkan *dashboard* interaktif publik menggunakan **Streamlit**. *Dashboard* ini berfungsi untuk menjawab permasalahan "ketidakmampuan mengukur risiko" dengan memberikan visualisasi edukatif (*exploratory & explanatory analysis*) kepada pengguna. Pengguna dapat melihat secara langsung bagaimana korelasi antara *screen time* yang berlebih atau defisit jam tidur dapat meningkatkan probabilitas *burnout*.
3.  **Integrasi Pipeline Data dan Rekomendasi:** Menyiapkan alur data (*data wrangling* dan *feature engineering*) yang terstandardisasi dan bersih dari bias (*data leakage*) agar siap dikonsumsi oleh model *Deep Learning*. Hasil akhir dari sistem tidak hanya memberikan output tingkat risiko, tetapi juga menghasilkan rekomendasi gaya hidup *actionable* (seperti target pengurangan *screen time* atau penambahan jam olahraga) guna mencegah mahasiswa jatuh ke fase *burnout* yang lebih parah.

# 1. IMPORT LIBRARY

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

sns.set_theme(style="whitegrid")

# 2. DATA LOADING

In [2]:
df = pd.read_csv(
    "Data/Raw_Responses/raw_google_form_export.csv",
    skiprows=[1],
    encoding="latin-1" 
)
df.head()

,Timestamp,1. Age\nExample: 23 [Restricted to 19ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½,2. Gender\nExample: Male,3. Study_Hours (per day) \nExample: 3,4. Class_Attendance (%) \nExample: 85,5. Tuition (Private Coaching) \nExample: Yes,6. Exam_Frequency (per semester) \nExample: 4,7. Assignment_Load (1ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½,8. Sleep_Hours (per day) \nExample: 6,9. Physical_Exercise (hours per week) \nExample: 2,10. Social_Media_Use (hours per day) \nExample: 3,11. Screen_Time (hours per day) \nExample: 6,12. Family_Income_Level \nExample: Middle,13. Peer_Pressure (1ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½,14. Family_Support (1ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½,15. Anxiety_Level (1ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½,16. University_Type \nExample: Public University,17. Stress_Score (1ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï¿½ï,18. Stress_Level \nExample: Moderate
0,2025/10/01 12:36:28 AM GMT+6,24,Female,8,43,Yes,6,8,5,Yes,6,8,Medium,3,7,8,National University,22,High
1,2025/10/01 01:12:57 AM GMT+6,21,Female,4,56,Yes,3,6,5,Yes,7,9,Low,2,5,3,Private University,16,Medium
2,2025/10/01 01:49:26 AM GMT+6,23,Female,4,46,No,8,6,8,No,5,6,Medium,8,6,5,Private University,16,Medium
3,2025/10/01 02:25:55 AM GMT+6,20,Male,8,46,No,5,1,6,Yes,0,7,Medium,5,6,2,National University,1,Low
4,2025/10/01 03:02:23 AM GMT+6,19,Male,2,95,No,5,2,7,No,3,6,Low,2,6,1,National University,4,Low


In [3]:
# Rename kolom setelah load — supaya bisa dipakai dari Section 3
df.columns = [
    "Timestamp","Age","Gender","Study_Hours","Class_Attendance",
    "Tuition","Exam_Frequency","Assignment_Load",
    "Sleep_Hours","Physical_Exercise","Social_Media_Use",
    "Screen_Time","Family_Income_Level","Peer_Pressure",
    "Family_Support","Anxiety_Level","University_Type",
    "Stress_Score","Stress_Level"
]

# 3. DATA UNDERSTANDING

In [4]:
print("Shape:", df.shape)
print("\nMissing:\n", df.isnull().sum())
print("\nNegative:\n", (df.select_dtypes(include="number") < 0).sum())
print("\nDuplicate:", df.duplicated().sum())
print("\nDtypes:\n", df.dtypes)

Shape: (2999, 19)

Missing:
 Timestamp              0
Age                    0
Gender                 0
Study_Hours            0
Class_Attendance       0
Tuition                0
Exam_Frequency         0
Assignment_Load        0
Sleep_Hours            0
Physical_Exercise      0
Social_Media_Use       0
Screen_Time            0
Family_Income_Level    0
Peer_Pressure          0
Family_Support         0
Anxiety_Level          0
University_Type        0
Stress_Score           0
Stress_Level           0
dtype: int64

Negative:
 Age                   0
Study_Hours           0
Class_Attendance      0
Exam_Frequency        0
Assignment_Load       0
Sleep_Hours           0
Social_Media_Use      0
Screen_Time           0
Peer_Pressure         0
Family_Support        0
Anxiety_Level         0
Stress_Score        110
dtype: int64

Duplicate: 0

Dtypes:
 Timestamp              object
Age                     int64
Gender                 object
Study_Hours             int64
Class_Attendance        in

In [5]:
print(df.info())
print(df.describe())
print("Duplicate:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Timestamp            2999 non-null   object
 1   Age                  2999 non-null   int64 
 2   Gender               2999 non-null   object
 3   Study_Hours          2999 non-null   int64 
 4   Class_Attendance     2999 non-null   int64 
 5   Tuition              2999 non-null   object
 6   Exam_Frequency       2999 non-null   int64 
 7   Assignment_Load      2999 non-null   int64 
 8   Sleep_Hours          2999 non-null   int64 
 9   Physical_Exercise    2999 non-null   object
 10  Social_Media_Use     2999 non-null   int64 
 11  Screen_Time          2999 non-null   int64 
 12  Family_Income_Level  2999 non-null   object
 13  Peer_Pressure        2999 non-null   int64 
 14  Family_Support       2999 non-null   int64 
 15  Anxiety_Level        2999 non-null   int64 
 16  Univer

In [6]:
#DISTRIBUSI TARGET
df["Stress_Level"].value_counts()

Stress_Level
Medium    1430
Low       1243
High       326
Name: count, dtype: int64

In [7]:
df["Stress_Level"].value_counts(normalize=True) * 100

Stress_Level
Medium    47.682561
Low       41.447149
High      10.870290
Name: proportion, dtype: float64

In [8]:
#CEK CATEGORICAL DISTRIBUTION
for col in ["Gender", "University_Type", "Family_Income_Level", "Tuition"]:
    print(f"\n{col}")
    print(df[col].value_counts())


Gender
Gender
Female    1538
Male      1461
Name: count, dtype: int64

University_Type
University_Type
Public University      1003
National University     999
Private University      997
Name: count, dtype: int64

Family_Income_Level
Family_Income_Level
Medium    1333
Low       1058
High       608
Name: count, dtype: int64

Tuition
Tuition
No     1544
Yes    1455
Name: count, dtype: int64


In [9]:
#CEK KORELASI
df.corr(numeric_only=True)["Screen_Time"].sort_values(ascending=False)

Screen_Time         1.000000
Stress_Score        0.494516
Class_Attendance    0.029559
Sleep_Hours         0.018414
Anxiety_Level       0.018224
Exam_Frequency      0.014956
Social_Media_Use    0.014897
Assignment_Load     0.004322
Age                -0.012583
Peer_Pressure      -0.026959
Family_Support     -0.034991
Study_Hours        -0.035367
Name: Screen_Time, dtype: float64

In [10]:
#GROUP ANALYSIS
df.groupby("Stress_Level")[[
    "Sleep_Hours",
    "Screen_Time",
    "Study_Hours",
    "Peer_Pressure"
]].mean().round(2)

,Sleep_Hours,Screen_Time,Study_Hours,Peer_Pressure
Stress_Level,,,,
High,5.88,8.65,4.29,4.83
Low,6.90,4.37,4.51,5.00
Medium,6.34,6.64,4.51,5.01


# 4. DATA CLEANING

In [11]:
# Drop kolom tidak penting
df = df.drop(columns=["Timestamp"])

In [12]:
# Convert numeric
num_cols = [
    "Age","Study_Hours","Class_Attendance","Exam_Frequency",
    "Assignment_Load","Sleep_Hours","Social_Media_Use",
    "Screen_Time","Peer_Pressure","Family_Support",
    "Anxiety_Level","Stress_Score"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [13]:
# Handle invalid negative values
num_cols = df.select_dtypes(include="number").columns

df[num_cols] = df[num_cols].mask(df[num_cols] < 0)

In [14]:
# Clean categorical
cat_cols = ["Gender","Tuition","University_Type","Family_Income_Level","Stress_Level"]

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

In [15]:
# Filter umur
df = df[(df["Age"] >= 19) & (df["Age"] <= 24)]

# Drop leakage
df = df.drop(columns=["Stress_Score","Anxiety_Level"])

In [16]:
df.head()

,Age,Gender,Study_Hours,Class_Attendance,Tuition,Exam_Frequency,Assignment_Load,Sleep_Hours,Physical_Exercise,Social_Media_Use,Screen_Time,Family_Income_Level,Peer_Pressure,Family_Support,University_Type,Stress_Level
0,24,Female,8,43,Yes,6,8,5,Yes,6,8,Medium,3,7,National University,High
1,21,Female,4,56,Yes,3,6,5,Yes,7,9,Low,2,5,Private University,Medium
2,23,Female,4,46,No,8,6,8,No,5,6,Medium,8,6,Private University,Medium
3,20,Male,8,46,No,5,1,6,Yes,0,7,Medium,5,6,National University,Low
4,19,Male,2,95,No,5,2,7,No,3,6,Low,2,6,National University,Low


In [17]:
# Range Validation
# Filter umur sudah dilakukan di Cell 21 — hanya validasi range kolom lain
df = df[
    (df["Class_Attendance"].between(0, 100)) &
    (df["Exam_Frequency"].between(1, 10)) &
    (df["Assignment_Load"].between(1, 10)) &
    (df["Peer_Pressure"].between(1, 10)) &
    (df["Family_Support"].between(1, 10))
]

In [18]:
binary_map = {
    "Yes": 1,
    "No": 0
}

df["Physical_Exercise"] = df["Physical_Exercise"].map(binary_map)
df["Tuition"] = df["Tuition"].map(binary_map)

In [19]:
print(df["Physical_Exercise"].unique())
print(df["Tuition"].unique())

[1 0]
[1 0]


In [20]:
income_map = {"Low": 0, "Medium": 1, "High": 2}
df["Family_Income_Level"] = df["Family_Income_Level"].map(income_map)


In [21]:
print(df["Tuition"].dtype)
print(df["Family_Income_Level"].dtype)

int64
int64


In [22]:
# Drop missing & duplicate
df = df.dropna()
df = df.drop_duplicates()

# 5. EXPLORATORY DATA ANALYSIS (EDA) 

🔑 Key Findings:
- Dataset menunjukkan ketidakseimbangan kelas, dengan kategori High stress hanya sekitar 10%
- Screen time memiliki hubungan paling jelas dengan tingkat stress
- Mahasiswa dengan stress tinggi memiliki screen time paling tinggi
- Durasi tidur berbanding terbalik dengan stress
- Study hours tidak menunjukkan pengaruh signifikan terhadap stress
- Korelasi linear antar variabel relatif lemah, menunjukkan kemungkinan hubungan non-linear

## Distribusi Target

In [ ]:
#CEK DISTRIBUSI NUMERIK
df.hist(figsize=(12,8))
plt.tight_layout()
plt.show()

In [ ]:
order = ["Low", "Medium", "High"]
 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
 
# Count plot
sns.countplot(x="Stress_Level", data=df, order=order, hue="Stress_Level", legend=False, palette="Set2", ax=axes[0])
axes[0].set_title("Distribusi Tingkat Stress (Count)")
axes[0].set_xlabel("Stress Level")
axes[0].set_ylabel("Jumlah Mahasiswa")
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)
 
# Pie chart
counts = df["Stress_Level"].value_counts().reindex(order)
axes[1].pie(counts, labels=order, autopct='%1.1f%%',
            colors=sns.color_palette("Set2", 3), startangle=90)
axes[1].set_title("Proporsi Tingkat Stress")
 
plt.tight_layout()
plt.show()
 
print("Distribusi Stress Level:")
print(df["Stress_Level"].value_counts(normalize=True).reindex(order).mul(100).round(1).astype(str) + "%")

## Distribusi Fitur Numerik

### Korelasi

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = df.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=0.5,
    annot_kws={"size": 8}
)
plt.title("Correlation Matrix (Fitur Numerik)", fontsize=13)
plt.tight_layout()
plt.show()

## Business Questions

### BQ1: Faktor apa yang paling berpengaruh terhadap stress?

In [ ]:
# Encode Stress_Level menjadi numerik untuk korelasi
stress_map = {"Low": 0, "Medium": 1, "High": 2}
df["Stress_Num"] = df["Stress_Level"].map(stress_map)
 
corr_stress = (
    df.corr(method='spearman', numeric_only=True)["Stress_Num"]
    .drop("Stress_Num")
    .sort_values(key=abs, ascending=True)
)
 
plt.figure(figsize=(8, 6))
colors = ["#E74C3C" if v > 0 else "#3498DB" for v in corr_stress]
corr_stress.plot(kind="barh", color=colors)
plt.axvline(0, color="black", linewidth=0.8, linestyle="--")
plt.title("BQ1 — Korelasi Fitur terhadap Stress Level\n(merah = positif, biru = negatif)", fontsize=12)
plt.xlabel("Korelasi Pearson")
plt.tight_layout()
plt.show()
 
print("\nInsight BQ1:")
top_pos = corr_stress[corr_stress > 0].nlargest(3)
top_neg = corr_stress[corr_stress < 0].nsmallest(3)
print(f"  Faktor yang MENINGKATKAN stress: {', '.join(top_pos.index.tolist())}")
print(f"  Faktor yang MENURUNKAN stress : {', '.join(top_neg.index.tolist())}")
 
df.drop(columns=["Stress_Num"], inplace=True)

### BQ2: Apakah tidur < 6 jam berhubungan dengan stress lebih tinggi?

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot
sns.boxplot(
    x="Stress_Level",
    y="Sleep_Hours",
    data=df,
    order=order,
    hue="Stress_Level",
    legend=False,
    palette="Set2",
    ax=axes[0]
)

# Mean tiap kategori
group_means = df.groupby("Stress_Level")["Sleep_Hours"].mean().reindex(order)

# Garis mean
for i, mean in enumerate(group_means):
    axes[0].hlines(
        mean,
        i - 0.4,
        i + 0.4,
        colors="red",
        linestyles="--",
        linewidth=1.5
    )

# Custom legend
mean_line = Line2D(
    [0], [0],
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="Mean"
)

axes[0].legend(handles=[mean_line], loc="upper center")

axes[0].set_title("BQ2 — Sleep Hours per Stress Level")
axes[0].set_xlabel("Stress Level")
axes[0].set_ylabel("Jam Tidur per Hari")

# Bar rata-rata
group_means.plot(
    kind="bar",
    color=sns.color_palette("Set2", 3),
    ax=axes[1],
    edgecolor="white"
)

axes[1].axhline(
    6,
    color="red",
    linestyle="--",
    linewidth=1.5,
    label="6 jam (batas ideal)"
)

axes[1].set_title("Rata-rata Sleep Hours per Stress Level")
axes[1].set_xlabel("Stress Level")
axes[1].set_ylabel("Rata-rata Jam Tidur")
axes[1].set_xticklabels(order, rotation=0)

axes[1].legend()

plt.tight_layout()
plt.show()

print("\nInsight BQ2:")
print(
    df.groupby("Stress_Level")["Sleep_Hours"]
    .mean()
    .reindex(order)
    .round(2)
    .to_string()
)

low_sleep = (df["Sleep_Hours"] < 6).mean() * 100

print(f"\n{low_sleep:.1f}% mahasiswa tidur kurang dari 6 jam per hari")

### BQ3: Apakah screen time tinggi berhubungan dengan stress lebih besar?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
# Boxplot
sns.boxplot(x="Stress_Level", y="Screen_Time", data=df, order=order,
            hue="Stress_Level", legend=False, palette="Set2", ax=axes[0])
axes[0].set_title("BQ3 — Screen Time per Stress Level")
axes[0].set_xlabel("Stress Level")
axes[0].set_ylabel("Screen Time (jam/hari)")
 
# Scatter screen time vs sleep, hue=stress
sns.scatterplot(
    x="Screen_Time", y="Sleep_Hours", hue="Stress_Level",
    hue_order=order, data=df, palette="Set2", alpha=0.55,
    s=40, ax=axes[1]
)
axes[1].set_title("Screen Time vs Sleep Hours\n(warna = Stress Level)")
axes[1].set_xlabel("Screen Time (jam/hari)")
axes[1].set_ylabel("Jam Tidur")
 
plt.tight_layout()
plt.show()
 
print("\n Insight BQ3:")
print(df.groupby("Stress_Level")["Screen_Time"].mean().reindex(order).round(2).to_string())

### BQ4: Pengaruh Peer Pressure & Family Support terhadap stress?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
sns.boxplot(x="Stress_Level", y="Peer_Pressure", data=df, order=order,
            hue="Stress_Level", legend=False, palette="Reds", ax=axes[0])
axes[0].set_title("BQ4a — Peer Pressure per Stress Level")
axes[0].set_xlabel("Stress Level")
axes[0].set_ylabel("Peer Pressure (skala 1–10)")
 
sns.boxplot(x="Stress_Level", y="Family_Support", data=df, order=order,
            hue="Stress_Level", legend=False, palette="Blues", ax=axes[1])
axes[1].set_title("BQ4b — Family Support per Stress Level")
axes[1].set_xlabel("Stress Level")
axes[1].set_ylabel("Family Support (skala 1–10)")
 
plt.tight_layout()
plt.show()
 
print("\nInsight BQ4:")
print(df.groupby("Stress_Level")[["Peer_Pressure", "Family_Support"]].mean().reindex(order).round(2).to_string())

### BQ5: Apakah mahasiswa yang ikut les (Tuition) lebih stres?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
# Countplot
sns.countplot(x="Tuition", hue="Stress_Level", data=df,
              hue_order=order, palette="Set2", ax=axes[0])
axes[0].set_title("BQ5 — Stress Level berdasarkan Status Les")
axes[0].set_xlabel("Tuition (1=Ya, 0=Tidak)")
axes[0].set_ylabel("Jumlah Mahasiswa")
 
# Stacked bar proporsi
tuition_stress = (
    df.groupby(["Tuition", "Stress_Level"])
    .size()
    .unstack("Stress_Level")
    .reindex(columns=order)
    .div(df.groupby("Tuition").size(), axis=0)
    * 100
)
tuition_stress.plot(
    kind="bar", stacked=True, colormap="Set2", ax=axes[1], edgecolor="white"
)
axes[1].set_title("Proporsi Stress Level per Status Les (%)")
axes[1].set_xlabel("Tuition (1=Ya, 0=Tidak)")
axes[1].set_ylabel("Proporsi (%)")
axes[1].set_xticklabels(["Tidak Les", "Les"], rotation=0)
axes[1].legend(title="Stress Level", loc="upper right")
 
plt.tight_layout()
plt.show()
 
print("\nInsight BQ5:")
print(tuition_stress.round(1).to_string())
 

### BQ6 — Study Hours vs Stress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(x="Stress_Level", y="Study_Hours", data=df, order=order,
            hue="Stress_Level", legend=False, palette="Set2", ax=axes[0])
axes[0].set_title("BQ6 — Study Hours per Stress Level")
axes[0].set_xlabel("Stress Level")
axes[0].set_ylabel("Jam Belajar per Hari")

group_means = df.groupby("Stress_Level")["Study_Hours"].mean().reindex(order)
group_means.plot(kind="bar", color=sns.color_palette("Set2", 3),
                 ax=axes[1], edgecolor="white")
axes[1].set_title("Rata-rata Study Hours per Stress Level")
axes[1].set_xlabel("Stress Level")
axes[1].set_ylabel("Rata-rata Jam Belajar")
axes[1].set_xticklabels(order, rotation=0)

plt.tight_layout()
plt.show()

print("\nInsight BQ6:")
print(df.groupby("Stress_Level")["Study_Hours"].mean().reindex(order).round(2).to_string())

### BQ7 — Gender vs Stress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(x="Gender", hue="Stress_Level", data=df,
              hue_order=order, palette="Set2", ax=axes[0])
axes[0].set_title("BQ7 — Stress Level berdasarkan Gender (Count)")
axes[0].set_xlabel("Gender")
axes[0].set_ylabel("Jumlah Mahasiswa")

gender_stress = (
    df.groupby(["Gender", "Stress_Level"])
    .size()
    .unstack("Stress_Level")
    .reindex(columns=order)
    .div(df.groupby("Gender").size(), axis=0)
    * 100
)
gender_stress.plot(kind="bar", stacked=True, colormap="Set2",
                   ax=axes[1], edgecolor="white")
axes[1].set_title("Proporsi Stress Level per Gender (%)")
axes[1].set_xlabel("Gender")
axes[1].set_ylabel("Proporsi (%)")
axes[1].set_xticklabels(gender_stress.index, rotation=0)
axes[1].legend(title="Stress Level", loc="upper right")

plt.tight_layout()
plt.show()

print("\nInsight BQ7:")
print(gender_stress.round(1).to_string())

###  RINGKASAN INSIGHT EDA
1. BQ1 — Faktor dominan stress:
    - (korelasi positif) Screen time, Exam frequency, Assignment load 
    - (korelasi negatif) Sleep hours, Family support, Physical exercise 
2. BQ2 — Sleep & Stress:
    - Low stress tidur 6.90 jam, High stress hanya 5.88 jam
    - 33.2% mahasiswa tidur kurang dari 6 jam per hari 
3. BQ3 — Screen Time & Stress:
    - Low=4.37 jam, Medium=6.64 jam, High=8.65 jam/hari 
    - Semakin tinggi stress, screen time naik drastis
4. BQ4 — Peer Pressure & Family Support: 
    - Family support turun tajam: Low=5.99, Med=4.58, High=3.31
    - Peer pressure relatif stabil di semua kategori (~5.0)
5. BQ5 — Tuition & Stress:
    - Mahasiswa les sedikit lebih banyak di High stress
    - (11.4% vs 10.4%) — perbedaan tidak terlalu signifikan 
6. BQ6 — Study Hours & Stress:
    - Jam belajar hampir sama di semua kategori (~4.5 jam)
    - Study hours TIDAK berpengaruh signifikan terhadap stress 
7. BQ7 — Gender & Stress: 
    - Proporsi High stress sama antara pria & wanita (10.9%)
    - Gender TIDAK menunjukkan perbedaan pola stress

# 6. FEATURE ENGINEERING

In [ ]:
df["Sleep_Deficit"] = (7 - df["Sleep_Hours"]).clip(lower=0)

df["Screen_to_Sleep"] = df["Screen_Time"] / (df["Sleep_Hours"] + 1)

df.head()

In [ ]:
# Productivity vs distraction
df["Study_to_Screen"] = df["Study_Hours"] / (df["Screen_Time"] + 1)

# Pressure vs support
df["Pressure_Index"] = (df["Peer_Pressure"] + 1) / (df["Family_Support"] + 1)

# Lifestyle balance
df["Activity_Balance"] = df["Physical_Exercise"] / (df["Screen_Time"] + 1)

In [ ]:
df["Lifestyle_Risk"] = (
    df["Screen_Time"] +
    df["Peer_Pressure"] -
    df["Sleep_Hours"] -
    df["Physical_Exercise"]
)

# Geser nilai negatif agar nilai minimum menjadi 0
min_risk = df["Lifestyle_Risk"].min()
if min_risk < 0:
    df["Lifestyle_Risk"] = df["Lifestyle_Risk"] + abs(min_risk)


In [ ]:
df["Academic_Pressure"] = (
    df["Assignment_Load"] +
    df["Exam_Frequency"] +
    df["Study_Hours"]
)

In [ ]:
df["Rest_Efficiency"] = df["Sleep_Hours"] / (df["Screen_Time"] + 1)

In [ ]:
order = ["Low", "Medium", "High"]

new_features = [
    "Sleep_Deficit",
    "Screen_to_Sleep",
    "Study_to_Screen",
    "Pressure_Index",
    "Activity_Balance",
    "Lifestyle_Risk",
    "Academic_Pressure",
    "Rest_Efficiency"
]

for col in new_features:
    sns.boxplot(x="Stress_Level", y=col, data=df, order=order, hue="Stress_Level", legend=False)
    plt.title(f"{col} vs Stress Level")
    plt.show()

In [ ]:
import numpy as np

# --- Validasi Akhir Section 6 ---
print("=== Shape ===")
print("Rows:", len(df), "| Cols:", df.shape[1])

print("\n=== Missing Values ===")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "Tidak ada missing value")

print("\n=== Negative Values ===")
neg = (df.select_dtypes(include="number") < 0).sum()
print(neg[neg > 0].to_string() or "Tidak ada nilai negatif")

print("\n=== Infinity Values ===")
inf_vals = np.isinf(df.select_dtypes(include="number")).sum()
print(inf_vals[inf_vals > 0].to_string() or "Tidak ada nilai infinity")

print("\n=== Dtypes ===")
print(df.dtypes.to_string())


## A/B Testing: Gender vs Lifestyle Risk
Menguji hipotesis apakah ada perbedaan rata-rata skor `Lifestyle_Risk` yang signifikan antara mahasiswa Pria dan Wanita menggunakan Independent T-Test.

In [ ]:
from scipy import stats

# Pisahkan data berdasarkan Gender
male_risk = df[df['Gender'] == 'Male']['Lifestyle_Risk']
female_risk = df[df['Gender'] == 'Female']['Lifestyle_Risk']

# Lakukan Independent T-Test
t_stat, p_value = stats.ttest_ind(male_risk.dropna(), female_risk.dropna(), equal_var=False)

print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("\nKesimpulan: Tolak H0. Terdapat perbedaan signifikan rata-rata Lifestyle_Risk antara pria dan wanita.")
else:
    print("\nKesimpulan: Gagal tolak H0. TIDAK ADA perbedaan signifikan rata-rata Lifestyle_Risk antara pria dan wanita.")

# 7. PREPROCESSING

## Split

In [ ]:
from sklearn.model_selection import train_test_split

# Drop feature yang kurang relevan berdasarkan EDA
df = df.drop(columns=["Study_Hours", "Gender"], errors="ignore")

X = df.drop(columns=["Stress_Level"])
# Ubah target string menjadi numerik (0, 1, 2)
stress_map = {"Low": 0, "Medium": 1, "High": 2}
y = df["Stress_Level"].map(stress_map)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Encoding + Column Setup

In [ ]:
num_cols = df.drop(columns=["Stress_Level"]).select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = df.drop(columns=["Stress_Level"]).select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

## Scaling + Encoding (ColumnTransformer)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

transformers_list = [("num", StandardScaler(), num_cols)]
if cat_cols:
    transformers_list.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols))

preprocessor = ColumnTransformer(
    transformers=transformers_list,
    remainder="drop"
)

## Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor)
])

## Transform

In [ ]:
X_train_processed = pipeline.fit_transform(X_train)
X_test_processed = pipeline.transform(X_test)

# Penanganan Class Imbalance dengan SMOTE
try:
    from imblearn.over_sampling import SMOTE
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn", "-q"])
    from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_processed, y_train = smote.fit_resample(X_train_processed, y_train)

print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)


In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Total features:", len(feature_names))

## Save Artifact

In [ ]:
import joblib

joblib.dump(pipeline, "preprocessing_pipeline.pkl")

In [ ]:
np.save("X_train.npy", X_train_processed)
np.save("X_test.npy", X_test_processed)

pd.Series(y_train, name="Stress_Level").to_csv("y_train.csv", index=False)
pd.Series(y_test, name="Stress_Level").to_csv("y_test.csv", index=False)

# 8. MODELING (Anak AI)

# 9. EVALUATION (Anak AI)

---
## 📌 Catatan untuk Tim AI Engineer (Farel & Fadhil)

1. **Validation Split:** Data latih (`X_train.npy`, `y_train.csv`) yang diberikan ini memiliki porsi 80% dari total dataset. Saat melakukan *training* menggunakan arsitektur Keras/TensorFlow, **wajib tambahkan parameter `validation_split=0.2`** di dalam pemanggilan `model.fit()` agar memonitor proses belajar per *epoch* dan mencegah *overfitting*.
2. **Metrik Evaluasi:** Karena data *testing* (`X_test.npy`, `y_test.csv`) dibiarkan natural (tidak di-SMOTE/tetap *imbalanced* untuk menjaga kondisi *real-world*), harap **gunakan F1-Score (Macro)** sebagai metrik evaluasi utama, bukan sekadar *Accuracy*.